In [ ]:
# ============================================================
# STEP 1: DATASET CLEANING
# PCOS Classification Project
# ============================================================

import pandas as pd
import numpy as np

# ── 1. Load ──────────────────────────────────────────────────
df = pd.read_csv('PCOS_extended_dataset.csv')
print(f"Original shape: {df.shape}")
print(f"Original class distribution:\n{df['PCOS (Y/N)'].value_counts()}\n")

# ── 2. Drop identifier columns ───────────────────────────────
id_cols = ['Sl. No', 'Patient File No.']
df = df.drop(columns=id_cols)
print(f"After dropping ID columns: {df.shape}")

# ── 3. Drop ultrasound features (Rotterdam Criterion 3) ──────
# Justification: these features ARE the diagnostic criterion.
# Excluding them makes the research question honest:
# "Can non-imaging features triage PCOS within a clinical cohort?"
ultrasound_cols = [
    'Follicle No. (L)',
    'Follicle No. (R)',
    'Avg. F size (L) (mm)',
    'Avg. F size (R) (mm)',
    'Endometrium (mm)'
]
df = df.drop(columns=ultrasound_cols)
print(f"After dropping ultrasound features: {df.shape}")
print(f"Ultrasound features removed: {ultrasound_cols}\n")

# ── 4. Force all non-target columns to numeric ───────────────
target = 'PCOS (Y/N)'
before_rows = len(df)

for col in df.columns:
    if col == target:
        continue
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna()
print(f"After fixing non-numeric values: {len(df)} rows "
      f"(removed {before_rows - len(df)} corrupted rows)")

# ── 5. Remove clinically impossible values ───────────────────
# FSH > 100 mIU/mL is physiologically impossible in PCOS context
# LH > 100 mIU/mL is physiologically impossible in PCOS context
# These are data entry errors (e.g., 50.52 entered as 5052)
before_rows = len(df)
df = df[df['FSH(mIU/mL)'] < 100]
df = df[df['LH(mIU/mL)'] < 100]
print(f"After removing impossible FSH/LH values: {len(df)} rows "
      f"(removed {before_rows - len(df)} rows)")

# ── 6. Check for duplicates ───────────────────────────────────
dupes = df.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")
if dupes > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed. Remaining rows: {len(df)}")

# ── 7. Final summary ─────────────────────────────────────────
print("\n" + "="*50)
print("CLEAN DATASET SUMMARY")
print("="*50)
print(f"Final shape:     {df.shape}")
print(f"Features:        {df.shape[1] - 1}")
print(f"Target:          {target}")
print(f"\nClass distribution:")
counts = df[target].value_counts()
print(f"  Non-PCOS (0):  {counts[0]} ({counts[0]/len(df)*100:.1f}%)")
print(f"  PCOS (1):      {counts[1]} ({counts[1]/len(df)*100:.1f}%)")
print(f"\nMissing values:  {df.isnull().sum().sum()}")
print(f"\nFeature list:")
for i, col in enumerate([c for c in df.columns if c != target]):
    print(f"  {i+1:2d}. {col}")

# ── 8. Save clean dataset ────────────────────────────────────
df.to_csv('PCOS_clean.csv', index=False)
print(f"\n✓ Clean dataset saved as 'PCOS_clean.csv'")
print(f"✓ Use this file for ALL future steps — never the original")

Original shape: (2000, 44)
Original class distribution:
PCOS (Y/N)
0    1392
1     608
Name: count, dtype: int64

After dropping ID columns: (2000, 42)
After dropping ultrasound features: (2000, 37)
Ultrasound features removed: ['Follicle No. (L)', 'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)']

After fixing non-numeric values: 1989 rows (removed 11 corrupted rows)
After removing impossible FSH/LH values: 1979 rows (removed 10 rows)

Duplicate rows: 0

CLEAN DATASET SUMMARY
Final shape:     (1979, 37)
Features:        36
Target:          PCOS (Y/N)

Class distribution:
  Non-PCOS (0):  1386 (70.0%)
  PCOS (1):      593 (30.0%)

Missing values:  0

Feature list:
   1.  Age (yrs)
   2. Weight (Kg)
   3. Height(Cm) 
   4. BMI
   5. Blood Group
   6. Pulse rate(bpm) 
   7. RR (breaths/min)
   8. Hb(g/dl)
   9. Cycle(R/I)
  10. Cycle length(days)
  11. Marraige Status (Yrs)
  12. Pregnant(Y/N)
  13. No. of abortions
  14.   I   beta-HCG(mIU/mL)
  15.